# **DDL SEMANTICA**

In [0]:
%sql
CREATE OR REPLACE VIEW products.semantica.gold_data AS
    select
    dp.product_name,
    dm.market_name,
    dm.department,
    dm.city,
    fpp.price_currency,
    dt.start_market_date,
    fpp.start_date_price,
    dt.end_market_date,
    fpp.end_date_price, 
    dt.market_year,
    dt.market_month,
    dt.market_quarter,
    dt.market_semester,
    dt.week_number,
    fpp.is_current
    from
    products.gold.fact_product_prices fpp
        inner join products.gold.dim_product dp
        on fpp.product_id = dp.product_id
        inner join products.gold.dim_time dt
        on fpp.date_id = dt.date_id
        inner join products.gold.dim_market dm
        on fpp.market_id = dm.market_id

In [0]:
%sql
-- 1
--Actual Price Difference between start date and end date
-- Ver diferencia de Precios
CREATE OR REPLACE VIEW products.semantica.vw_price_weekly_variation AS
SELECT
  dp.product_name,
  dm.market_name,
  dm.department,
  dm.city,
  fpp.price_currency,
  dt.start_market_date,
  fpp.start_date_price,
  dt.end_market_date,
  fpp.end_date_price,
  (fpp.end_date_price - fpp.start_date_price) AS price_difference,
  ROUND(
    ((fpp.end_date_price - fpp.start_date_price) / NULLIF(fpp.start_date_price, 0)) * 100,
    2
  ) AS percentage_variation,
  round(avg(fpp.start_date_price) over (partition by dp.product_name), 2) as avg_product_price,
  fpp.start_date_price - avg_product_price as diff_avg
FROM
  products.gold.fact_product_prices fpp
    INNER JOIN products.gold.dim_product dp
      ON fpp.product_id = dp.product_id
    INNER JOIN products.gold.dim_time dt
      ON fpp.date_id = dt.date_id
    INNER JOIN products.gold.dim_market dm
      ON fpp.market_id = dm.market_id
WHERE
  fpp.is_current = TRUE
ORDER BY
  dm.department,
  dm.city,
  dm.market_name,
  dp.product_name

In [0]:
%sql
-- 2
-- Actual Highest Market All Products Price by Department
-- Ver cual es la canasta basica mas cara x mercado y depto
CREATE OR REPLACE VIEW products.semantica.vw_market_price_ranking AS
SELECT
  dm.market_name,
  dm.department,
  dm.city,
  fpp.price_currency,
  COUNT(*) AS total_products_tracked,
  ROUND(AVG(fpp.end_date_price), 2) AS avg_market_price
FROM
  products.gold.fact_product_prices fpp
    INNER JOIN products.gold.dim_product dp
      ON fpp.product_id = dp.product_id
    INNER JOIN products.gold.dim_time dt
      ON fpp.date_id = dt.date_id
    INNER JOIN products.gold.dim_market dm
      ON fpp.market_id = dm.market_id
WHERE
  fpp.is_current = TRUE
GROUP BY
  dm.market_name,
  dm.city,
  dm.department,
  fpp.price_currency
order by
  dm.department,
  dm.city,
  avg_market_price desc

In [0]:
%sql
-- 3
-- REVISAR
-- Evolution Historical Prices
CREATE OR REPLACE VIEW products.semantica.vw_product_historical_trends AS
SELECT
  dt.market_year,
  dt.week_number,
  case
    when dt.week_number <= 9 then concat(dt.market_year,'0', CAST(dt.week_number as varchar(2)))
    else concat(dt.market_year, CAST(dt.week_number as varchar(2)))
  end as year_week,
  dt.market_month,
  dp.product_name,
  dm.market_name,
  dm.department,
  dm.city,
  fpp.price_currency,
  ROUND(AVG(fpp.end_date_price), 2) AS avg_weekly_price,
  ROUND(AVG(fpp.start_date_price), 2) AS avg_start_price
FROM
  products.gold.fact_product_prices fpp
    INNER JOIN products.gold.dim_product dp
      ON fpp.product_id = dp.product_id
    INNER JOIN products.gold.dim_time dt
      ON fpp.date_id = dt.date_id
    INNER JOIN products.gold.dim_market dm
      ON fpp.market_id = dm.market_id
GROUP BY
  dt.market_year,
  dt.week_number,
  dt.market_month,
  dp.product_name,
  dm.market_name,
  dm.department,
  dm.city,
  fpp.price_currency
order by
  dt.market_year ASC,
  dt.week_number ASC,
  year_week,
  dp.product_name ASC

In [0]:
%sql
-- 4
-- Analysis per quarter,
CREATE OR REPLACE VIEW products.semantica.vw_seasonal_price_analysis AS
SELECT
  dt.market_year,
  dt.market_semester,
  dt.market_quarter,
  dp.product_name,
  dm.department,
  dm.city,
  fpp.price_currency,
  ROUND(AVG(fpp.end_date_price), 2) AS avg_quarterly_price,
  ROUND(MIN(fpp.start_date_price), 2) AS min_price_in_quarter,
  ROUND(MAX(fpp.end_date_price), 2) AS max_price_in_quarter
FROM
  products.gold.fact_product_prices fpp
    INNER JOIN products.gold.dim_product dp
      ON fpp.product_id = dp.product_id
    INNER JOIN products.gold.dim_time dt
      ON fpp.date_id = dt.date_id
    INNER JOIN products.gold.dim_market dm
      ON fpp.market_id = dm.market_id
GROUP BY
  dt.market_year,
  dt.market_semester,
  dt.market_quarter,
  dp.product_name,
  dm.department,
  dm.city,
  fpp.price_currency
ORDER BY dt.market_year ASC, dt.market_quarter ASC;

In [0]:
%sql
--5
--Actual Highest Market Price by Department (HACE UNA WINDOWS FUNCTION DE PRECIO PROMEDIO y DIFERENCIA POR PRECIO EN CADA PRODUCTO Y MERCADO)

/*
SELECT
  dp.product_name,
  dm.market_name,
  dm.department,
  dm.city,
  fpp.price_currency,
  dt.start_market_date,
  fpp.start_date_price,
  dt.end_market_date,
  fpp.end_date_price,
  round(avg(fpp.start_date_price) over(partition by dp.product_name),2) as avg_price,
  fpp.start_date_price - avg_price as diff
FROM
  products.gold.fact_product_prices fpp
    INNER JOIN products.gold.dim_product dp
      ON fpp.product_id = dp.product_id
    INNER JOIN products.gold.dim_time dt
      ON fpp.date_id = dt.date_id
    INNER JOIN products.gold.dim_market dm
      ON fpp.market_id = dm.market_id
WHERE fpp.is_current = TRUE
ORDER BY dp.product_name,fpp.start_date_price desc
*/